# Qubit spectroscopy intrinsic width

In [9]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from qblox_lab.config.device import save_device_configuration
from qblox_lab.config.hardware import create_hardware_agent
from qblox_lab.experiments.cal07_qubit_spectroscopy_intrinsic_width import QubitSpectroscopyIntrinsicWidth

## Run parameters

In [ ]:
HARDWARE_CONFIG = Path(r"..\config\hw_config.json")
DEVICE_CONFIG = Path(r"..\config\dut_config.json")
FLUX_CONFIG = Path(r"..\config\flux_config.json")  # If None, leave the existing hardware flux unchanged
OUTPUT_DIR = Path("data")

QUBITS = ["q0"]
FREQUENCY_CENTER = 5.4e9
FREQUENCY_WIDTH = 10e6
FREQUENCY_POINTS = 1001
MAXIMUM_BRANCH_WIDTH = 800e6
REPETITIONS = 10
DRIVE_AMPLITUDES = [0.002, 0.005, 0.01, 0.02]
# DRIVE_AMPLITUDES = np.arange(start = 0.1, stop = 0.5, step = 0.1)   # Alternative sweep-like method
DRIVE_DURATION = 100e-6

READOUT_AMPLITUDE = None
DRIVE_OUTPUT_ATTENUATION = None
READOUT_OUTPUT_ATTENUATION = None
READOUT_INPUT_ATTENUATION = None
RESTORE_DRIVE_LO_FREQUENCY = None
TIMEOUT = 300
CREATE_DUMMY_CONNECTIONS = False

PLOT_RESULTS = True
SAVE_FITTED_DEVICE = Path(r"..\config\dut_config_upd.json")  # Use None if parameter persistence is not required


## Hardware and experiment

In [18]:
hardware_agent = create_hardware_agent(
    hardware_configuration=HARDWARE_CONFIG,
    device_configuration=DEVICE_CONFIG,
    output_dir=OUTPUT_DIR,
    create_dummy_connections=CREATE_DUMMY_CONNECTIONS,
)

experiment = QubitSpectroscopyIntrinsicWidth(
    hardware_agent=hardware_agent,
    qubits=QUBITS,
    flux_config=FLUX_CONFIG,
)

## Branch plan

In [19]:
branches = experiment.plan_branches(
    frequency_center=FREQUENCY_CENTER,
    frequency_width=FREQUENCY_WIDTH,
    frequency_points=FREQUENCY_POINTS,
    maximum_branch_width=MAXIMUM_BRANCH_WIDTH,
)
for branch in branches:
    print(
        f"Branch {branch.index + 1}: "
        f"{branch.start / 1e9:.6f}-{branch.stop / 1e9:.6f} GHz, "
        f"LO {branch.lo_frequency / 1e9:.6f} GHz, "
        f"{branch.points} points"
    )

Branch 1: 5.395000-5.405000 GHz, LO 5.400000 GHz, 1001 points


## Measurement

In [20]:
dataset = experiment.run_measurement(
    frequency_center=FREQUENCY_CENTER,
    frequency_width=FREQUENCY_WIDTH,
    frequency_points=FREQUENCY_POINTS,
    maximum_branch_width=MAXIMUM_BRANCH_WIDTH,
    repetitions=REPETITIONS,
    drive_amplitudes=DRIVE_AMPLITUDES,
    drive_duration=DRIVE_DURATION,
    readout_amplitude=READOUT_AMPLITUDE,
    drive_output_attenuation=DRIVE_OUTPUT_ATTENUATION,
    readout_output_attenuation=READOUT_OUTPUT_ATTENUATION,
    readout_input_attenuation=READOUT_INPUT_ATTENUATION,
    restore_drive_lo_frequency=RESTORE_DRIVE_LO_FREQUENCY,
    timeout=TIMEOUT,
)
dataset

c:\Users\yi.huang\anaconda3\envs\QSE\lib\site-packages\qblox_scheduler\backends\qblox\compiler_abc.py:285: UserWarning: Exact capabilities for ISA version 2.1 not found, using capabilities for version 2.2.
  warnings.warn(
c:\Users\yi.huang\anaconda3\envs\QSE\lib\site-packages\qblox_scheduler\backends\qblox\compiler_abc.py:285: UserWarning: Exact capabilities for ISA version 2.1 not found, using capabilities for version 2.2.
  warnings.warn(
c:\Users\yi.huang\anaconda3\envs\QSE\lib\site-packages\qblox_scheduler\backends\qblox\compiler_abc.py:285: UserWarning: Exact capabilities for ISA version 2.1 not found, using capabilities for version 2.2.
  warnings.warn(
c:\Users\yi.huang\anaconda3\envs\QSE\lib\site-packages\qblox_scheduler\backends\qblox\compiler_abc.py:285: UserWarning: Exact capabilities for ISA version 2.1 not found, using capabilities for version 2.2.
  warnings.warn(
c:\Users\yi.huang\anaconda3\envs\QSE\lib\site-packages\qblox_scheduler\backends\qblox\compiler_abc.py:285: U

<xarray.Dataset> Size: 160kB
Dimensions:             (acq_index_S21_q0: 4004)
Coordinates:
    drive_amplitude_q0  (acq_index_S21_q0) float64 32kB 0.1 0.1 0.1 ... 0.4 0.4
    frequency_q0        (acq_index_S21_q0) float64 32kB 5.395e+09 ... 5.405e+09
  * acq_index_S21_q0    (acq_index_S21_q0) int64 32kB 0 1 2 3 ... 4001 4002 4003
Data variables:
    S21_q0              (acq_index_S21_q0) complex128 64kB (0.015115760183334...
Attributes:
    tuid:     20260811-161300-021-bb5d18

## 2D Plot

In [ ]:
for qubit in QUBITS:
    frequencies = np.asarray(dataset[f"frequency_{qubit}"].values).ravel()
    drive_amplitudes = np.asarray(dataset[f"drive_amplitude_{qubit}"].values).ravel()
    transmission = np.asarray(dataset[f"S21_{qubit}"].values).ravel()

    valid = (
        np.isfinite(frequencies)
        & np.isfinite(drive_amplitudes)
        & np.isfinite(transmission)
    )
    unique_frequencies = np.unique(frequencies[valid])
    unique_amplitudes = np.unique(drive_amplitudes[valid])
    frequency_indices = np.searchsorted(unique_frequencies, frequencies[valid])
    amplitude_indices = np.searchsorted(unique_amplitudes, drive_amplitudes[valid])

    transmission_sum = np.zeros(
        (unique_amplitudes.size, unique_frequencies.size),
        dtype=complex,
    )
    sample_count = np.zeros(transmission_sum.shape, dtype=int)
    np.add.at(
        transmission_sum,
        (amplitude_indices, frequency_indices),
        transmission[valid],
    )
    np.add.at(sample_count, (amplitude_indices, frequency_indices), 1)
    averaged_transmission = np.divide(
        transmission_sum,
        sample_count,
        out=np.full(transmission_sum.shape, np.nan + 0j),
        where=sample_count > 0,
    )

    figure, axis = plt.subplots()
    image = axis.pcolormesh(
        unique_frequencies / 1e9,
        unique_amplitudes,
        np.abs(averaged_transmission),
        shading="auto",
    )
    axis.set_xlabel("Qubit drive frequency (GHz)")
    axis.set_ylabel("Normalized drive amplitude")
    axis.set_title(f"Power-dependent qubit spectroscopy: {qubit}")
    figure.colorbar(image, ax=axis, label="|S21| (V)")
    figure.tight_layout()
plt.show()

## Analysis

In [24]:
ANALYSIS_DRIVE_AMPLITUDE = 0.1

In [25]:
results = experiment.analysis(drive_amplitude=ANALYSIS_DRIVE_AMPLITUDE)
print(f"Analyzed drive amplitude: {ANALYSIS_DRIVE_AMPLITUDE:.9g}")
for qubit, result in results.items():
    if result.success:
        print(
            f"{qubit}: f01 = {result.frequency / 1e9:.9f} GHz, "
            f"linewidth = {result.linewidth / 1e6:.3f} MHz"
        )
    else:
        print(f"{qubit}: qubit-spectroscopy fit failed")

Analyzed drive amplitude: 0.1
q0: f01 = 5.398952133 GHz, linewidth = 6.174 MHz


In [ ]:
if PLOT_RESULTS:
    experiment.plot()

## Optional: device update

In [ ]:
if SAVE_FITTED_DEVICE is not None:
    experiment.update_device()
    saved_path = save_device_configuration(
        hardware_agent.quantum_device,
        SAVE_FITTED_DEVICE,
    )
    print(f"Updated device configuration saved to {saved_path}")